# Formativa 1 — Rain in Australia

In [1]:
from pathlib import Path

import pandas as pd

SEMILLA = 42
ruta_csv = Path("data/raw/weatherAUS.csv")
if Path.cwd().name == "notebooks":
    ruta_csv = Path("..") / ruta_csv

datos = pd.read_csv(ruta_csv)
filas, columnas = datos.shape
print(f"{filas:,}".replace(",", ".") + f" filas y {columnas} columnas")


145.460 filas y 23 columnas


## 1. Perfilado inicial del conjunto de datos

Se inspeccionan los tipos inferidos por `pandas`, la cantidad de valores válidos, el porcentaje de datos faltantes (`NA`), la cardinalidad y el rango observado en cada una de las 23 columnas de `weatherAUS.csv`.

In [ ]:
resumen_cols = []
for col in datos.columns:
    serie = datos[col]
    n_na = int(serie.isna().sum())
    pct_na = round(100 * n_na / len(datos), 2)
    n_uniq = int(serie.nunique(dropna=True))
    if pd.api.types.is_numeric_dtype(serie):
        rango = f"{serie.min()} .. {serie.max()}"
    else:
        muestra = serie.dropna().unique()[:4]
        rango = ", ".join(map(str, muestra))
    resumen_cols.append({
        "Variable": col,
        "Dtype pandas": str(serie.dtype),
        "Faltantes": n_na,
        "% Faltantes": pct_na,
        "Valores únicos": n_uniq,
        "Rango / Ejemplos": rango,
    })

perfil_df = pd.DataFrame(resumen_cols)
perfil_df

## 2. Clasificación de las variables según su tipo estadístico

A partir de la naturaleza física de las mediciones del *Bureau of Meteorology* y de su dominio observado en las 145.460 filas, las 23 variables se clasifican según su tipo estadístico, escala de medición y rol analítico:

| Tipo estadístico | Escala de medición | Variables | Dominio / Unidad observada | Rol analítico |
|---|---|---|---|---|
| **Temporal (fecha)** | Intervalo calendario | `Date` | Fechas diarias (`2007-11-01` a `2017-06-25`) | Índice temporal; permite derivar año, mes y estacionalidad |
| **Cualitativa nominal** | Nominal | `Location` | 49 estaciones meteorológicas de Australia | Factor de agrupación espacial / heterogeneidad geográfica |
| **Cualitativa nominal (direccional)** | Nominal / Circular | `WindGustDir`, `WindDir9am`, `WindDir3pm` | 16 puntos cardinales (`N`, `NNE`, ..., `NNW`) | Explicativa categórica (con estructura angular periódica) |
| **Cualitativa dicotómica (binaria)** | Nominal binaria | `RainToday` | `No` / `Yes` (`Yes` si `Rainfall` > 1 mm) | Explicativa binaria (persistencia diaria) |
| **Cualitativa dicotómica (binaria)** | Nominal binaria | `RainTomorrow` | `No` / `Yes` | **Variable objetivo** (clasificación binaria) |
| **Cuantitativa discreta** | Discreta (ordinal en oktas) | `Cloud9am`, `Cloud3pm` | Enteros `0` a `9` oktas (9 = cielo oscurecido) | Explicativa de cobertura nubosa |
| **Cuantitativa continua** | Intervalo | `MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm` | °C (`-8,5` a `48,1`) | Explicativas térmicas |
| **Cuantitativa continua** | Intervalo | `Pressure9am`, `Pressure3pm` | hPa (`977,1` a `1041,0`) | Explicativas barométricas |
| **Cuantitativa continua** | Razón (no negativa) | `Rainfall`, `Evaporation`, `Sunshine` | mm (`0`–`371`, `0`–`145`) y horas (`0`–`14,5`) | Explicativas hidrológicas y de radiación solar |
| **Cuantitativa continua** | Razón (acotada) | `Humidity9am`, `Humidity3pm` | Porcentaje `0` a `100` % | Explicativas higrométricas |
| **Cuantitativa continua*** | Razón (no negativa) | `WindGustSpeed`, `WindSpeed9am`, `WindSpeed3pm` | km/h (`6`–`135`, `0`–`130`, `0`–`87`) | Explicativas de intensidad del viento |

> *Nota:* Las velocidades del viento se registran como valores enteros en el CSV, pero conceptualmente representan magnitudes físicas continuas.

**Observación sobre datos faltantes (`NA`):** Los porcentajes más altos de ausencia se concentran en `Sunshine` (48,01 %), `Evaporation` (43,17 %), `Cloud3pm` (40,81 %) y `Cloud9am` (38,42 %), seguidos por `Pressure9am` (10,36 %) y `Pressure3pm` (10,33 %). Esto responde a diferencias de instrumentación entre estaciones (`Location`), por lo que no se deben asumir como faltantes completamente aleatorios (MCAR).

## 3. Posibles relaciones entre variables en un contexto estadístico

En el análisis estadístico del dataset destacan cuatro ejes de relación:

1. **Dependencia lineal y colinealidad entre variables numéricas:**
   - **Variables térmicas:** `MaxTemp` y `Temp3pm` presentan una correlación positiva muy alta (r = 0,985), al igual que `MinTemp` y `Temp9am` (r = 0,902). Incluir todas juntas en modelos lineales sin selección o regularización introduce multicolinealidad.
   - **Pares intra-día (`9am` vs. `3pm`):** `Pressure9am` y `Pressure3pm` están fuertemente correlacionadas (r = 0,961), mientras que `Humidity9am`–`Humidity3pm` (r = 0,667) y `WindGustSpeed`–`WindSpeed3pm` (r = 0,686) muestran asociaciones positivas moderadas-altas. Resulta útil evaluar diferencias intra-día como `Pressure3pm - Pressure9am` (caída barométrica) o la amplitud térmica `MaxTemp - MinTemp`.
   - **Relaciones inversas por física atmosférica:** Mayor nubosidad vespertina (`Cloud3pm`) reduce las horas de sol (`Sunshine`, r = -0,704), y mayor humedad relativa a las 15:00 (`Humidity3pm`) se asocia negativamente con `Sunshine` (r = -0,629) y `Temp3pm` (r = -0,558).
2. **Asociación con la variable objetivo (`RainTomorrow`):**
   - Los días previos a lluvia (`RainTomorrow = Yes`) muestran en promedio mayor `Humidity3pm` (68,80 % vs. 46,51 %), mayor `Cloud3pm` (6,36 vs. 3,92 oktas), menos `Sunshine` (4,47 vs. 8,55 horas), menor `Pressure3pm` (1012,30 vs. 1016,11 hPa) y mayor `Rainfall` (6,14 vs. 1,27 mm).
   - `Rainfall` posee una fuerte asimetría positiva y gran masa en cero (mediana = 0 mm, máximo = 371 mm), por lo que conviene complementarla con estadísticos no paramétricos (correlación de Spearman, mediana y RIC).
3. **Persistencia temporal (`RainToday` → `RainTomorrow`):**
   - La lluvia presenta autocorrelación temporal: cuando `RainToday = No`, llueve al día siguiente en 15,19 % de los casos (16.604 de 109.332); cuando `RainToday = Yes`, la proporción sube a 46,41 % (14.597 de 31.455).
4. **Heterogeneidad espacial (`Location`) y estacionalidad (`Date`):**
   - Las 49 estaciones cubren climas muy distintos (árido, templado, tropical), por lo que `Location` y el mes derivado de `Date` actúan como variables de contexto que modifican tanto los niveles basales como la probabilidad de lluvia.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

vars_num = [
    "MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
    "WindGustSpeed", "WindSpeed9am", "WindSpeed3pm",
    "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
    "Cloud9am", "Cloud3pm", "Temp9am", "Temp3pm"
]

corr_pearson = datos[vars_num].corr(method="pearson")

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    corr_pearson,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    annot_kws={"size": 7},
    ax=ax,
)
ax.set_title("Matriz de correlación de Pearson entre variables cuantitativas")
plt.tight_layout()
plt.show()

In [ ]:
# Comparación de variables clave según RainTomorrow
vars_clave = ["Humidity3pm", "Sunshine", "Cloud3pm", "Pressure3pm", "Temp3pm", "Rainfall"]
resumen_objetivo = (
    datos.dropna(subset=["RainTomorrow"])
    .groupby("RainTomorrow")[vars_clave]
    .agg(["count", "mean", "median"])
    .round(2)
)
display(resumen_objetivo)

# Tabla de contingencia RainToday vs RainTomorrow y prueba chi-cuadrado
tabla_lluvia = pd.crosstab(datos["RainToday"], datos["RainTomorrow"])
chi2, p_valor, _, _ = stats.chi2_contingency(tabla_lluvia)
display(tabla_lluvia)
display(pd.crosstab(datos["RainToday"], datos["RainTomorrow"], normalize="index").round(4) * 100)
print(f"Chi-cuadrado de independencia: chi2 = {chi2:.2f}, p-valor = {p_valor:.3e}")

In [ ]:
# Variación estacional y por ubicación
datos_tiempo = datos.assign(
    Mes=pd.to_datetime(datos["Date"]).dt.month,
    LlueveManana=(datos["RainTomorrow"] == "Yes").astype(float),
)
datos_tiempo.loc[datos["RainTomorrow"].isna(), "LlueveManana"] = float("nan")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

tasa_mes = datos_tiempo.groupby("Mes")["LlueveManana"].mean() * 100
tasa_mes.plot(kind="bar", color="steelblue", ax=axes[0])
axes[0].set_title("Tasa de lluvia al día siguiente (%) por mes")
axes[0].set_xlabel("Mes")
axes[0].set_ylabel("% RainTomorrow = Yes")

tasa_loc = (
    datos_tiempo.groupby("Location")["LlueveManana"]
    .mean()
    .sort_values()
    .mul(100)
)
extremos_loc = pd.concat([tasa_loc.head(5), tasa_loc.tail(5)])
extremos_loc.plot(kind="barh", color="teal", ax=axes[1])
axes[1].set_title("Estaciones con menor y mayor tasa de RainTomorrow = Yes")
axes[1].set_xlabel("% RainTomorrow = Yes")

plt.tight_layout()
plt.show()

## 4. Síntesis para etapas posteriores

- **Selección de pruebas:** Para variables asimétricas (`Rainfall`, `Evaporation`) es preferible usar medidas robustas (mediana, RIC) y pruebas no paramétricas o de remuestreo; para variables aproximadamente simétricas (`Temp3pm`, `Pressure3pm`), las aproximaciones paramétricas son adecuadas considerando el gran tamaño muestral (n > 130.000).
- **Control de colinealidad:** Evitar redundancia directa entre pares altamente correlacionados (`MaxTemp`/`Temp3pm` y `Pressure9am`/`Pressure3pm`), optando por una representante o por su diferencia.
- **Tratamiento de faltantes y estructura:** Dado que `Sunshine`, `Evaporation` y `Cloud*` faltan en bloque según la estación, cualquier imputación o modelado futuro debe considerar `Location` y `Date` para no introducir sesgo.